In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../../data/bpic17.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:LoanGoal": "string",
        "case:ApplicationType": "string",
        "Accepted": "string",
        "Selected": "string",
        "case:RequestedAmount": "float32",
        "FirstWithdrawalAmount": "float32",
        "NumberOfTerms": "float32",
        "MonthlyCost": "float32",
        "CreditScore": "float32",
        "OfferedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
0,Application_1000086665,2016-08-03 15:57:21.673,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Create Application,complete,User_1,0.000000e+00
1,Application_1000086665,2016-08-03 15:57:21.734,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Submitted,complete,User_1,6.100000e-02
2,Application_1000086665,2016-08-03 15:58:28.299,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Concept,complete,User_1,6.656500e+01
3,Application_1000086665,2016-08-05 13:57:07.419,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Accepted,complete,User_5,1.655191e+05
4,Application_1000086665,2016-08-05 13:59:57.320,True,0.0,5000.0,241.279999,22.0,5000.0,False,New credit,"Other, see explanation",5000.0,O_Create Offer,complete,User_5,1.699010e+02
5,Application_1000086665,2016-08-05 13:59:58.162,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Created,complete,User_5,8.420000e-01
6,Application_1000086665,2016-08-05 14:01:23.264,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Sent (mail and online),complete,User_5,8.510200e+01
7,Application_1000086665,2016-08-05 14:01:23.288,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Complete,complete,User_5,2.400000e-02
8,Application_1000086665,2016-09-05 06:00:36.710,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Cancelled,complete,User_1,2.649554e+06
9,Application_1000086665,2016-09-05 06:00:36.829,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Cancelled,complete,User_1,1.190000e-01


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Accepted', 'CreditScore', 'FirstWithdrawalAmount', 'MonthlyCost', 'NumberOfTerms', 'OfferedAmount', 'Selected', 'case:ApplicationType', 'case:LoanGoal', 'case:RequestedAmount', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:LoanGoal                  categorical    case     yes    ['Boat', 'Business goal', 'Car', ...]    N/A        data_derived        
case:ApplicationType           categorical    case     yes    ['Limit raise', 'New credit']            N/A        data_derived        
Accepted         

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    path = "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'A_Incomplete', 'A_Validating', 'O_Returned'}]

In [16]:
engine.branching_sets

[{'A_Complete',
  'A_Incomplete',
  'A_Validating',
  'O_Create Offer',
  'O_Created',
  'O_Returned',
  'O_Sent (mail and online)'},
 {'O_Create Offer', 'O_Created', 'O_Sent (mail and online)'},
 {'A_Denied', 'O_Refused'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic17-cf_seed777_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,Application_509718181,7,1,0,0.195162,0.390324,0.0,0.583333,0.176471,...,1.928282,0.176471,0.195162,0.0,0.390324,0.583333,0.973316,0.973316,1.000000,1.000000
1,0,Application_1424170205,9,1,0,0.115084,0.230168,0.0,0.458333,0.142857,...,1.462071,0.142857,0.115084,0.0,0.230168,0.458333,0.745797,0.745797,0.999999,0.999999
2,0,Application_542586744,10,1,3,0.121821,0.243642,0.0,0.520833,0.130435,...,1.736081,0.130435,0.121821,0.0,0.243642,0.520833,0.962991,0.962991,1.000000,1.000000
3,0,Application_1690291723,11,1,2,0.119336,0.238673,0.0,0.541667,0.280000,...,1.876613,0.280000,0.119336,0.0,0.238673,0.541667,0.935610,0.935610,1.000000,1.000000
4,0,Application_135180486,12,1,3,0.136346,0.272693,0.0,0.562500,0.111111,...,1.650354,0.111111,0.136346,0.0,0.272693,0.562500,0.840396,0.840396,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,48,Application_317628520,22,1,8,0.185207,0.370415,0.0,0.568627,0.617021,...,1.370856,0.617021,0.185207,0.0,0.370415,0.568627,0.000000,0.942812,0.000000,1.000000
375,48,Application_2144131288,23,1,5,0.163506,0.327013,0.0,0.490196,0.632653,...,1.286356,0.632653,0.163506,0.0,0.327013,0.490196,0.000000,0.973222,0.000000,1.000000
376,48,Application_2057503198,24,1,6,0.167429,0.334857,0.0,0.578431,0.372549,...,2.065913,0.372549,0.167429,0.0,0.334857,0.578431,0.947504,0.947504,1.000000,1.000000
377,48,Application_727907012,25,1,3,0.111178,0.222356,0.0,0.509804,0.622642,...,1.243623,0.622642,0.111178,0.0,0.222356,0.509804,0.000000,0.000000,0.000000,0.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,Application_509718181,7,1,0,0.433449,0.396309,0.470588,0.687500,0.117647,...,1.238596,0.117647,0.433449,0.470588,0.396309,0.687500,0.000000,0.000000,0.0,0.0
1,0,Application_1424170205,9,1,0,0.187617,0.257588,0.117647,0.520833,0.285714,...,0.994165,0.285714,0.187617,0.117647,0.257588,0.520833,0.000000,0.000000,0.0,0.0
2,0,Application_542586744,10,1,3,0.319916,0.345714,0.294118,0.687500,0.347826,...,1.784265,0.347826,0.319916,0.294118,0.345714,0.687500,0.429023,0.429023,0.0,0.0
3,0,Application_1690291723,11,1,2,0.438689,0.406789,0.470588,0.750000,0.400000,...,2.079674,0.400000,0.438689,0.470588,0.406789,0.750000,0.490985,0.490985,0.0,0.0
4,0,Application_135180486,12,1,3,0.414021,0.357453,0.470588,0.729167,0.444444,...,1.693379,0.444444,0.414021,0.470588,0.357453,0.729167,0.105747,0.105747,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,48,Application_317628520,22,1,8,0.184453,0.311763,0.057143,0.549020,0.617021,...,1.350494,0.617021,0.184453,0.057143,0.311763,0.549020,0.000000,0.000000,0.0,0.0
375,48,Application_2144131288,23,1,5,0.190837,0.324530,0.057143,0.519608,0.632653,...,1.343097,0.632653,0.190837,0.057143,0.324530,0.519608,0.000000,0.000000,0.0,0.0
376,48,Application_2057503198,24,1,6,0.236603,0.387492,0.085714,0.627451,0.470588,...,1.334643,0.470588,0.236603,0.085714,0.387492,0.627451,0.000000,0.000000,0.0,0.0
377,48,Application_727907012,25,1,3,0.122215,0.244430,0.000000,0.480392,0.622642,...,1.225249,0.622642,0.122215,0.000000,0.244430,0.480392,0.000000,0.000000,0.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()